In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import os
import json

d:\MIT-collect\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
ROOT_DATA_SOURCE = "../data/source_data"

In [6]:
ROOT_DATA_ARTIFACTS = "../data/artifacts/"

In [4]:
MEANINGLESS_NER_CATEGORIES = {
    "DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "CARDINAL", "ORDINAL"
}


In [5]:
def combine_smart_features(row):
    keywords = []
    if pd.notna(row["keywords"]) and isinstance(row["keywords"], str):
        for kw in row["keywords"].split("; "):
            try:
                keyword, _ = kw.split("|")
                keywords.append(keyword)
            except:
                pass

    ner = []
    if pd.notna(row["ner_entities"]) and isinstance(row["ner_entities"], str):
        for entity in row["ner_entities"].split("; "):
            try:
                word, label, _ = entity.split("|")
                if label not in MEANINGLESS_NER_CATEGORIES:
                    ner.append(word)
            except:
                pass
    return " ".join(keywords + ner)

# Фильтр по общей частоте

In [6]:
def filter_5_percent_of_words(df):
    # Подсчёт частоты слов в признаке
    word_freq = {}
    for text in df["smart_features"]:
        if pd.notna(text):
            for word in text.split():
                word_freq[word] = word_freq.get(word, 0) + 1

    # Определение порога для 5% самых частых слов
    total_words = sum(word_freq.values())
    threshold = total_words * 0.05

    # Выделение слов, которые входят в топ-5%
    top_words = set()
    cumulative_freq = 0
    for word, freq in sorted(word_freq.items(), key=lambda x: x[1], reverse=True):
        cumulative_freq += freq
        top_words.add(word)
        if cumulative_freq >= threshold:
            break

    # Фильтрация признаков, удаляя слова из топ-5%
    def filter_top_words(text):
        if pd.isna(text):
            return text
        return " ".join([word for word in text.split() if word not in top_words])

    df["filtered_smart_features"] = df["smart_features"].apply(filter_top_words)

# Фильтр по df

In [8]:
embeddings = []
def generate_embeddings(df, field, model, batch_size=32, output_path=None):
    # Проверка наличия уже сохранённых эмбеддингов для данного датасета
    if os.path.exists(output_path):
        print(f"Загрузка эмбеддингов из {output_path}...")
        return np.load(output_path)
    for i in tqdm(range(0, len(df), batch_size), desc="Эмбеддинги"):
        batch = df[field].tolist()[i:i+batch_size]
        batch_embeddings = model.encode(batch, show_progress_bar=False)
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)

# Данные без ORCID


In [16]:
INPUT_PATH = ROOT_DATA_ARTIFACTS + "/articles_without_orcid_filtered.csv"

In [17]:
OUTPUT_PATH = ROOT_DATA_ARTIFACTS + "/embeddings_smart_without_orcid.npy"

In [ ]:
# Генерация и сохранение эмбеддингов

# Загрузка данных
df = pd.read_csv(INPUT_PATH)

# Фильтрация NER: оставляем только осмысленные категории (исключаем численные и процентные)

print("Генерация объединённых признаков...")
df["smart_features"] = df.apply(combine_smart_features, axis=1)

# Генерация эмбеддингов с tqdm
print("\nГенерация эмбеддингов...")
model = SentenceTransformer('all-mpnet-base-v2')
embeddings = []
batch_size = 32

embeddings = generate_embeddings(df, "smart_features", model, batch_size, output_path=OUTPUT_PATH)
# Сохранение эмбеддингов
np.save(OUTPUT_PATH, embeddings)
print("Эмбеддинги сохранены в embeddings_smart.npy")

# Данные с ORCID

In [ ]:
INPUT_PATH = ROOT_DATA_ARTIFACTS + "/articles_with_orcid_filtered.csv"

In [ ]:
OUTPUT_PATH = ROOT_DATA_ARTIFACTS + "/embeddings_smart_with_orcid.npy"

In [ ]:
df = pd.read_csv(INPUT_PATH)
print("Генерация объединённых признаков...")
df["smart_features"] = df.apply(combine_smart_features, axis=1)
print("\nГенерация эмбеддингов...")
embeddings = generate_embeddings(df, "smart_features", model, batch_size, output_path=OUTPUT_PATH)
np.save(OUTPUT_PATH, embeddings)
print("Эмбеддинги сохранены в embeddings_smart_with_orcid.npy")

# Даннные ILC


In [4]:
INPUT_PATH = ROOT_DATA_SOURCE + "/ilc_subjects_level3.json"

In [5]:
OUTPUT_PATH = ROOT_DATA_ARTIFACTS + "/embeddings_ilc.npy"

In [6]:
TEXT_OUTPUT_PATH = "ilc_combined_texts.txt"

In [7]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# 1. Загрузка модели
model = SentenceTransformer('all-mpnet-base-v2')

# 2. Загрузка дерева ILC из JSON
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    tree = json.load(f)

# 3. Функция для рекурсивного сбора названий всех подклассов
def collect_subclass_names(node):
    """Возвращает список названий (label) всех дочерних узлов на любой глубине."""
    names = []
    for child in node.get('children', []):
        names.append(child['label'])
        names.extend(collect_subclass_names(child))
    return names

# 4. Формирование признаков для каждого корневого класса
data = []
for root in tree:
    description = root.get('description', '')
    subclass_names = collect_subclass_names(root)
    
    # Конкатенация: описание + пробел + все названия подклассов через пробел
    combined_text = description
    if subclass_names:
        combined_text += " " + " ".join(subclass_names)
    
    data.append({
        'notation': root['notation'],
        'label': root['label'],
        'combined_data_of_field': combined_text.strip()
    })

# 5. Создание DataFrame
df = pd.DataFrame(data)
print(f"Сформировано {len(df)} корневых классов.")
print("Пример признаков первого класса:")
print(df.iloc[0]['combined_data_of_field'][:200] + "...")

# 6. Функция генерации эмбеддингов
embeddings = []
batch_size = 32
print("Тексты, подаваемые на вход модели эмбеддингов (первые 300 символов):")
for idx, row in df.iterrows():
    notation = row['notation']
    label = row['label']
    text = row['combined_data_of_field']
    print(f"\n[{notation}] {label}")
    print("-" * 40)
    # Печатаем  обрезанный для читаемости
    if len(text) > 300:
        print(text[:300] + "... [обрезано]")
    else:
        print(text)
with open(TEXT_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for idx, row in df.iterrows():
        f.write(f"[{row['notation']}] {row['label']}\n")
        f.write("-" * 50 + "\n")
        f.write(row['combined_data_of_field'] + "\n\n")
print(f"\nПолные тексты сохранены в {TEXT_OUTPUT_PATH}")
# 7. Генерация эмбеддингов
embeddings_array = generate_embeddings(df, "combined_data_of_field", model, batch_size, output_path=OUTPUT_PATH)

# 8. Сохранение результатов
df['embedding'] = list(embeddings_array)
df.to_pickle(ROOT_DATA_ARTIFACTS + 'ilc_root_embeddings.pkl')
np.save(ROOT_DATA_ARTIFACTS + 'ilc_root_embeddings.npy', embeddings_array)

print(f"Эмбеддинги сохранены в {ROOT_DATA_ARTIFACTS + 'ilc_root_embeddings.pkl'}. Размерность: {embeddings_array.shape}")

d:\MIT-collect\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3400.10it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Сформировано 25 корневых классов.
Пример признаков первого класса:
world attributes of #aa elements subsets cosets individuals classes attributes of #aca the empty class relations attributes of #aea identity equality similarity inequality set membership parthood set ...
Тексты, подаваемые на вход модели эмбеддингов (первые 300 символов):

[a] forms
----------------------------------------
world attributes of #aa elements subsets cosets individuals classes attributes of #aca the empty class relations attributes of #aea identity equality similarity inequality set membership parthood set inclusion predications operations attributes of #aga homomorphisms sum difference direct-like operat... [обрезано]

[b] spacetime
----------------------------------------
present this that that yonder time time years present, past and future Big Bang time hundreds of thousands years hundreds of millions years billions years more than billions years space dimensions attributes of #bca straight lines surfac

NameError: name 'generate_embeddings' is not defined

# Embeddings without common words

In [22]:
OUTPUT_PATH = ROOT_DATA_ARTIFACTS + "embeddings_filtered.npy"
INPUT_PATH = ROOT_DATA_SOURCE + "/articles_without_orcid_filtered.csv"

In [23]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# Загрузка данных
df = pd.read_csv(INPUT_PATH)
# Фильтрация NER: оставляем только осмысленные категории (исключаем численные и процентные)

print("Генерация объединённых признаков...")
df["smart_features"] = df.apply(combine_smart_features, axis=1)
# Фильтрация топ-5% самых частых слов
filter_5_percent_of_words(df)
# Генерация эмбеддингов с tqdm
print("\nГенерация эмбеддингов...")
model = SentenceTransformer('all-mpnet-base-v2')
embeddings = []
batch_size = 32
# Путь для сохранения отфильтрованного датасета
FILTERED_CSV_PATH = ROOT_DATA_ARTIFACTS + "articles_filtered_for_ilc.csv"

# Если файл уже существует, можно пропустить фильтрацию
if not os.path.exists(FILTERED_CSV_PATH):
    from collections import defaultdict

    # Функция фильтрации по document frequency
    def filter_by_document_frequency(df, text_column, max_df_ratio=0.8, min_word_length=2):
        doc_words = []
        for text in df[text_column]:
            if pd.isna(text):
                words = []
            else:
                words = [w.lower() for w in str(text).split() if len(w) >= min_word_length]
            doc_words.append(set(words))
        
        df_counts = defaultdict(int)
        for words_set in doc_words:
            for w in words_set:
                df_counts[w] += 1
        
        total_docs = len(df)
        threshold = total_docs * max_df_ratio
        high_df_words = {w for w, cnt in df_counts.items() if cnt > threshold}
        print(f"Удалено слов с DF > {max_df_ratio*100:.0f}%: {len(high_df_words)}")
        
        def filter_text(text):
            if pd.isna(text):
                return ""
            words = [w for w in str(text).split() 
                     if len(w) >= min_word_length and w.lower() not in high_df_words]
            return " ".join(words)
        
        filtered_col = text_column + "_filtered"
        df[filtered_col] = df[text_column].apply(filter_text)
        return df, high_df_words

    # Применяем фильтрацию
    df, high_df_words = filter_by_document_frequency(df, "smart_features", max_df_ratio=0.10)
    
    # Сохраняем список стоп-слов для последующего использования в облаках слов
    import pickle
    with open(ROOT_DATA_ARTIFACTS + "high_df_words.pkl", "wb") as f:
        pickle.dump(high_df_words, f)
    
    # Сохраняем отфильтрованный датасет
    df.to_csv(FILTERED_CSV_PATH, index=False)
    print(f"Отфильтрованный датасет сохранён в {FILTERED_CSV_PATH}")
else:
    print(f"Отфильтрованный датасет уже существует: {FILTERED_CSV_PATH}")
embeddings = generate_embeddings(df, "smart_features_filtered", model, batch_size, output_path=OUTPUT_PATH)
# Сохранение эмбеддингов
np.save(OUTPUT_PATH, embeddings)
print(f"Эмбеддинги сохранены в {OUTPUT_PATH}")

Генерация объединённых признаков...

Генерация эмбеддингов...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5560.25it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Удалено слов с DF > 10%: 11
Отфильтрованный датасет сохранён в ../data/artifacts/articles_filtered_for_ilc.csv
Загрузка эмбеддингов из ../data/artifacts/embeddings_filtered.npy...
Эмбеддинги сохранены в ../data/artifacts/embeddings_filtered.npy


In [4]:
import os
import json
import requests
from rdflib import Graph
from rdflib.namespace import SKOS, RDF

ROOT_DATA_SOURCE = "../data/source_data"          # ваша папка
ILC_TTL_URL = "https://www.iskoi.org/ilc/2/skos/ILC2.ttl"
ILC_TTL_FILENAME = "ILC2.ttl"
OUTPUT_JSON = os.path.join(ROOT_DATA_SOURCE, "ilc_subjects_level7.json")

os.makedirs(ROOT_DATA_SOURCE, exist_ok=True)

# Скачивание TTL (если нет)
ttl_path = os.path.join(ROOT_DATA_SOURCE, ILC_TTL_FILENAME)
if not os.path.exists(ttl_path):
    print(f"Скачивание {ILC_TTL_URL} ...")
    r = requests.get(ILC_TTL_URL)
    r.raise_for_status()
    with open(ttl_path, 'wb') as f:
        f.write(r.content)
    print(f"Сохранён: {ttl_path}")
else:
    print(f"Файл уже есть: {ttl_path}")

# Парсинг RDF
g = Graph()
g.parse(ttl_path, format="turtle")

nodes = {}
for concept in g.subjects(RDF.type, SKOS.Concept):
    uri = str(concept)
    notation = None
    for o in g.objects(concept, SKOS.notation):
        notation = str(o)
        break
    if not notation:
        continue

    label = None
    for o in g.objects(concept, SKOS.prefLabel):
        label = str(o)
        break

    definition = None
    for o in g.objects(concept, SKOS.definition):
        definition = str(o)
        break

    nodes[uri] = {
        "uri": uri,
        "notation": notation,
        "label": label or notation,
        "description": definition or "",
        "children": set(),
        "parents": set()
    }

# Связи broader/narrower
for concept, broader in g.subject_objects(SKOS.broader):
    child_uri = str(concept)
    parent_uri = str(broader)
    if child_uri in nodes and parent_uri in nodes:
        nodes[child_uri]["parents"].add(parent_uri)
        nodes[parent_uri]["children"].add(child_uri)

def is_valid_subclass(uri):
    notation = nodes[uri]["notation"]
    return all(c.islower() or c.isdigit() for c in notation)

MAIN_NOTATIONS = set(chr(c) for c in range(ord('a'), ord('y') + 1))
root_uris = [uri for uri, data in nodes.items() if data["notation"] in MAIN_NOTATIONS]

def build_tree(uri, depth=1, max_depth=7):
    data = nodes[uri]
    node = {
        "notation": data["notation"],
        "label": data["label"],
        "description": data["description"],
        "children": []
    }
    if depth < max_depth:
        sorted_children = sorted(data["children"], key=lambda u: nodes[u]["notation"])
        for child_uri in sorted_children:
            if not is_valid_subclass(child_uri) or nodes[child_uri]["notation"] in MAIN_NOTATIONS:
                continue
            child_node = build_tree(child_uri, depth + 1, max_depth)
            node["children"].append(child_node)
    return node

tree = [build_tree(uri) for uri in sorted(root_uris, key=lambda u: nodes[u]["notation"])]

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(tree, f, ensure_ascii=False, indent=2)

print(f"Дерево ILC глубиной 7 сохранено в {OUTPUT_JSON}")

Файл уже есть: ../data/source_data\ILC2.ttl
Дерево ILC глубиной 7 сохранено в ../data/source_data\ilc_subjects_level7.json


In [8]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import os

# === Пути ===
ILC_JSON_PATH = ROOT_DATA_SOURCE + "/ilc_subjects_level7.json"   # новый файл с глубиной 7
OUTPUT_CENTROIDS_EMB = ROOT_DATA_ARTIFACTS + "/ilc_level4_centroids_deep.npy"
OUTPUT_CENTROIDS_DF = ROOT_DATA_ARTIFACTS + "/ilc_level4_centroids_deep.pkl"

# Загрузка дерева
with open(ILC_JSON_PATH, 'r', encoding='utf-8') as f:
    tree = json.load(f)

# === Функция сбора узлов на глубине 4 ===
def collect_nodes_at_depth(node, current_depth=1, target_depth=4):
    nodes = []
    if current_depth == target_depth:
        nodes.append(node)
    elif current_depth < target_depth:
        for child in node.get('children', []):
            nodes.extend(collect_nodes_at_depth(child, current_depth + 1, target_depth))
    return nodes

level4_nodes = []
for root in tree:
    level4_nodes.extend(collect_nodes_at_depth(root, current_depth=1, target_depth=4))

print(f"Найдено классов на глубине 4: {len(level4_nodes)}")

# === Функция сбора дочерних меток до относительной глубины 3 (т.е. уровни 5,6,7) ===
def collect_descendant_labels(node, current_rel_depth=0, max_rel_depth=3):
    """
    Рекурсивно собирает label всех потомков, начиная с узла node,
    до относительной глубины max_rel_depth включительно.
    current_rel_depth = 1 для прямых детей.
    """
    labels = []
    if current_rel_depth >= max_rel_depth:
        return labels
    for child in node.get('children', []):
        labels.append(child['label'])
        # рекурсивно собираем от потомков, увеличивая относительную глубину
        labels.extend(collect_descendant_labels(child, current_rel_depth + 1, max_rel_depth))
    return labels

# === Формирование текстов для каждого центра ===
data = []
for node in level4_nodes:
    description = node.get('description', '')
    label = node['label']
    # Собираем дочерние метки до 3 уровней вниз
    descendant_labels = collect_descendant_labels(node, current_rel_depth=0, max_rel_depth=3)
    
    combined = description
    if label:
        combined += " " + label
    if descendant_labels:
        combined += " " + " ".join(descendant_labels)
    combined = combined.strip()
    
    data.append({
        'notation': node['notation'],
        'label': label,
        'combined_text': combined
    })

df_centroids = pd.DataFrame(data)
print(f"Пример текста для первого центра:\n{df_centroids.iloc[0]['combined_text'][:500]}...")

# === Генерация эмбеддингов ===
model = SentenceTransformer('all-mpnet-base-v2')
batch_size = 32

def generate_embeddings(df, text_col, model, batch_size, output_path=None):
    if output_path and os.path.exists(output_path):
        print(f"Загрузка эмбеддингов из {output_path}")
        return np.load(output_path)
    embeddings = []
    for i in tqdm(range(0, len(df), batch_size), desc="Эмбеддинги центров"):
        batch = df[text_col].tolist()[i:i+batch_size]
        batch_emb = model.encode(batch, show_progress_bar=False)
        embeddings.extend(batch_emb)
    emb_array = np.array(embeddings)
    if output_path:
        np.save(output_path, emb_array)
    return emb_array

centroids_emb = generate_embeddings(df_centroids, "combined_text", model, batch_size,
                                    output_path=OUTPUT_CENTROIDS_EMB)

df_centroids.to_pickle(OUTPUT_CENTROIDS_DF)
print(f"Сохранено центров: {len(df_centroids)}")
print(f"Размерность эмбеддингов: {centroids_emb.shape}")


Найдено классов на глубине 4: 2252
Пример текста для первого центра:
structure-preserving subsets...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4278.32it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Эмбеддинги центров: 100%|██████████| 71/71 [00:07<00:00, 10.02it/s]

Сохранено центров: 2252
Размерность эмбеддингов: (2252, 768)


In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances

# === Настройки ===
ARTIFACTS_PATH = "../data/artifacts/"
DEMO_PATH = "../demo_artifacts_level4/"          # отдельная папка для результатов уровня 4
os.makedirs(DEMO_PATH, exist_ok=True)

# Выберите нужный файл с эмбеддингами статей
EMBEDDINGS_FILE = "embeddings_smart.npy"          # или "embeddings_filtered.npy"
ARTICLES_CSV = "/articles_without_orcid_filtered.csv"   # укажите правильный CSV со статьями

CENTROIDS_EMB = "ilc_level4_centroids_deep.npy"
CENTROIDS_DF = "ilc_level4_centroids_deep.pkl"

# === Загрузка данных ===
print("Загрузка эмбеддингов статей...")
embeddings_articles = np.load(ARTIFACTS_PATH + EMBEDDINGS_FILE)
print(f"Статей: {embeddings_articles.shape[0]}")

df_articles = pd.read_csv(ROOT_DATA_SOURCE + ARTICLES_CSV)
assert len(df_articles) == embeddings_articles.shape[0]

print("Загрузка центроидов ILC уровня 4...")
centroids_emb = np.load(ARTIFACTS_PATH + CENTROIDS_EMB)
df_centroids = pd.read_pickle(ARTIFACTS_PATH + CENTROIDS_DF)
print(f"Центроидов: {len(df_centroids)}")

# === Стандартизация ===
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_articles)
centroids_scaled = scaler.transform(centroids_emb)

# === Кластеризация (присвоение ближайшего центра) ===
n_clusters = len(centroids_scaled)
print(f"Число кластеров: {n_clusters}")

kmeans = KMeans(
    n_clusters=n_clusters,
    init=centroids_scaled,
    n_init=1,
    random_state=42,
    max_iter=1          # центры не двигаются
)
labels = kmeans.fit_predict(embeddings_scaled)

# Добавление результатов в DataFrame
df_articles["ilc_level4_cluster"] = labels
distances = pairwise_distances(embeddings_scaled, centroids_scaled, metric="cosine")
df_articles["min_distance_to_ilc_center"] = distances.min(axis=1)

# Добавление читаемой информации о классе ILC
df_articles["ilc_notation"] = df_articles["ilc_level4_cluster"].map(
    lambda i: df_centroids.iloc[i]["notation"]
)
df_articles["ilc_label"] = df_articles["ilc_level4_cluster"].map(
    lambda i: df_centroids.iloc[i]["label"]
)

# Сохранение результатов
output_csv = ARTIFACTS_PATH + "articles_with_ilc_level4.csv"
df_articles.to_csv(output_csv, index=False)
print(f"Результаты сохранены в {output_csv}")

Загрузка эмбеддингов статей...
Статей: 124282
Загрузка центроидов ILC уровня 4...
Центроидов: 2252
Число кластеров: 2252


d:\MIT-collect\venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (2221) found smaller than n_clusters (2252). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Результаты сохранены в ../data/artifacts/articles_with_ilc_level4.csv


In [15]:
import umap.umap_ as umap
import plotly.express as px

# Загрузка данных с метками уровня 4
df = pd.read_csv(ARTIFACTS_PATH + "articles_with_ilc_level4.csv")
print(f"Загружено статей: {len(df)}")

embeddings_articles = np.load(ARTIFACTS_PATH + EMBEDDINGS_FILE)

# Стандартизация (повторяем для воспроизводимости)
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_articles)

# UMAP проекция
print("Построение UMAP проекции...")
reducer = umap.UMAP(random_state=42)
embeddings_2d = reducer.fit_transform(embeddings_scaled)

df["umap_x"] = embeddings_2d[:, 0]
df["umap_y"] = embeddings_2d[:, 1]

# Читаемое имя класса
df["ilc_display_name"] = df["ilc_notation"] + " " + df["ilc_label"]

# Интерактивный scatter plot
fig = px.scatter(
    df,
    x="umap_x",
    y="umap_y",
    color="ilc_display_name",
    hover_name="title",
    hover_data={
        "ilc_notation": True,
        "ilc_label": True,
        "min_distance_to_ilc_center": ":.3f",
        "umap_x": False,
        "umap_y": False
    },
    title="Распределение статей по классам ILC уровня 4 (UMAP проекция)",
    color_discrete_sequence=px.colors.qualitative.Light24,
    labels={
        "ilc_notation": "Код ILC",
        "ilc_label": "Класс ILC",
        "min_distance_to_ilc_center": "Расст. до центра"
    }
)
fig.update_traces(marker=dict(size=6, opacity=0.7))

output_html = DEMO_PATH + "ilc_level4_umap.html"
fig.write_html(output_html)
print(f"График сохранён в {output_html}")

Загружено статей: 124282
Построение UMAP проекции...


d:\MIT-collect\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


График сохранён в ../demo_artifacts_level4/ilc_level4_umap.html


In [16]:
import os
import re
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Загрузка данных
df = pd.read_csv(ARTIFACTS_PATH + "articles_with_ilc_level4.csv")
print(f"Загружено статей: {len(df)}")

# Константы для парсинга (из вашего ноутбука)
MEANINGLESS_NER_CATEGORIES = {"DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "CARDINAL", "ORDINAL"}

def clean_word(word):
    if not word or word.lower() == 'nan':
        return None
    return word

def parse_ner_string(ner_str):
    if pd.isna(ner_str) or not isinstance(ner_str, str):
        return []
    words = []
    for entity in ner_str.split("; "):
        try:
            word, label, _ = entity.split("|")
            if label not in MEANINGLESS_NER_CATEGORIES:
                cleaned = clean_word(word.lower())
                if cleaned:
                    words.append(cleaned)
        except ValueError:
            continue
    return words

def parse_keywords_string(kw_str):
    if pd.isna(kw_str) or not isinstance(kw_str, str):
        return []
    words = []
    for kw in kw_str.split("; "):
        try:
            word, _ = kw.split("|")
            cleaned = clean_word(word.lower())
            if cleaned:
                words.append(cleaned)
        except ValueError:
            continue
    return words

def generate_wordcloud(text, title, output_path):
    if not text or len(text.strip()) == 0:
        print(f"Пропуск {output_path}: нет текста для облака.")
        return
    wordcloud = WordCloud(
        width=1200,
        height=800,
        background_color='white',
        colormap='viridis',
        max_words=200,
        relative_scaling=0.5,
        random_state=42
    ).generate(text)
    plt.figure(figsize=(12, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(title, fontsize=20, pad=20)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()

# Создаём папку для облаков
clouds_dir = os.path.join(DEMO_PATH, "wordclouds_level4")
os.makedirs(clouds_dir, exist_ok=True)

# Группировка по классам ILC уровня 4
for (notation, label), group in df.groupby(['ilc_notation', 'ilc_label']):
    # Очищаем имя папки от недопустимых символов
    safe_label = re.sub(r'[^\w\-_ ]', '', label).replace(' ', '_').replace('/', '_')
    class_dir = os.path.join(clouds_dir, f"{notation}_{safe_label}")
    os.makedirs(class_dir, exist_ok=True)
    
    ner_words = []
    kw_words = []
    for _, row in group.iterrows():
        ner_words.extend(parse_ner_string(row['ner_entities']))
        kw_words.extend(parse_keywords_string(row['keywords']))
    
    ner_text = " ".join(ner_words)
    kw_text = " ".join(kw_words)
    combined_text = ner_text + " " + kw_text
    
    class_title = f"{notation} – {label}"
    
    generate_wordcloud(
        ner_text,
        f"NER entities: {class_title}",
        os.path.join(class_dir, "ner_wordcloud.png")
    )
    generate_wordcloud(
        kw_text,
        f"Keywords: {class_title}",
        os.path.join(class_dir, "keywords_wordcloud.png")
    )
    generate_wordcloud(
        combined_text,
        f"NER + Keywords: {class_title}",
        os.path.join(class_dir, "combined_wordcloud.png")
    )
    
    print(f"Облака слов сохранены для {notation} ({label}) в {class_dir}")

Загружено статей: 124282
Облака слов сохранены для aauc (structure-preserving subsets) в ../demo_artifacts_level4/wordclouds_level4\aauc_structure-preserving_subsets
Облака слов сохранены для aaud (divisor subsets) в ../demo_artifacts_level4/wordclouds_level4\aaud_divisor_subsets
Облака слов сохранены для aaul (lateral subsets) в ../demo_artifacts_level4/wordclouds_level4\aaul_lateral_subsets
Облака слов сохранены для aaun (independent subsets) в ../demo_artifacts_level4/wordclouds_level4\aaun_independent_subsets
Облака слов сохранены для aaus (substructures) в ../demo_artifacts_level4/wordclouds_level4\aaus_substructures
Облака слов сохранены для acai (intensionals) в ../demo_artifacts_level4/wordclouds_level4\acai_intensionals
Облака слов сохранены для acan (occasionality) в ../demo_artifacts_level4/wordclouds_level4\acan_occasionality
Облака слов сохранены для aeab (arity) в ../demo_artifacts_level4/wordclouds_level4\aeab_arity
Облака слов сохранены для aead (cardinality) в ../demo_

MemoryError: Unable to allocate 2.88 MiB for an array with shape (512, 738) and data type uint64